In [70]:
#run this everytime an edit is made in utils.py so that we can use the helper functions in this Python Notebook
import importlib
import utils
importlib.reload(utils)

<module 'utils' from '/Users/matthewliew/FINM 375-a/matthew-work/hw-2/utils.py'>

In [52]:
import pandas as pd
import numpy as np

#import rate data
rate_data = pd.read_excel("fiderivs_2025-03-10.xlsx", sheet_name="rate curves")
display(rate_data.head())

#import rate tree
rate_tree = pd.read_excel("fiderivs_2025-03-10.xlsx", sheet_name="rate tree")
display(rate_tree.head())

,tenor,swap rates,spot rates,discounts,forwards,flat vols
0,0.25,0.042192,0.042192,0.989562,NaN,NaN
1,0.50,0.040930,0.040923,0.979848,0.039655,0.146300
2,0.75,0.039760,0.039744,0.970775,0.037387,0.168563
3,1.00,0.038833,0.038808,0.962115,0.036001,0.190826
4,1.25,0.037868,0.037830,0.954026,0.033918,0.221870


,state,0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,...,2.5,2.75,3,3.25,3.5,3.75,4,4.25,4.5,4.75
0,0,0.041971,0.042342,0.042850,0.046844,0.051540,0.066325,0.080096,0.096150,0.108175,...,0.141028,0.158682,0.175308,0.201235,0.241151,0.277688,0.306393,0.333953,0.373457,0.413041
1,1,NaN,0.036579,0.037018,0.038854,0.041221,0.049622,0.058944,0.070184,0.078976,...,0.104662,0.118764,0.131738,0.151063,0.180726,0.208062,0.230372,0.252785,0.284441,0.315840
2,2,NaN,NaN,0.031980,0.032226,0.032969,0.037126,0.043378,0.051230,0.057659,...,0.077673,0.088888,0.098997,0.113400,0.135442,0.155895,0.173212,0.191345,0.216642,0.241513
3,3,NaN,NaN,NaN,0.026729,0.026368,0.027777,0.031923,0.037395,0.042096,...,0.057644,0.066527,0.074393,0.085127,0.101505,0.116807,0.130235,0.144839,0.165004,0.184678
4,4,NaN,NaN,NaN,NaN,0.021089,0.020782,0.023493,0.027296,0.030733,...,0.042780,0.049792,0.055904,0.063904,0.076071,0.087520,0.097921,0.109635,0.125674,0.141218


In [53]:
NOTIONAL = 100

### 1. Pricing the Components

## 1.1.
A floater with no credit spread trades at par. Record the floater value.

In [54]:
#Since a floater with no creadit spread is just the notional
floater_value = NOTIONAL

## 1.2.
Price the floor (strike = 2%) and the cap (strike = 5%).

Use the flat vol at the 5-year tenor from the curves table.

Report both prices.



In [55]:
floor_strike = 0.02
cap_strike = 0.05
five_year_flat_vol = rate_data.loc[rate_data["tenor"] == 5, "flat vols"].iloc[0]

cap_price = utils.price_cap(rate_data, 5, cap_strike, implied_vol=five_year_flat_vol)
print(f"Cap Price: {cap_price}")

floor_price = utils.price_floor(rate_data, 5, floor_strike, implied_vol=five_year_flat_vol)
print(f"Floor Price: {floor_price}")


Cap Price: 0.9773140911242582
Floor Price: 0.24491092604572093


## 1.3.
Combine the three instruments into the replicating portfolio:
- +1 Floater
- +1 Floor (2%)
- -1 Cap (5%)

Report the NAV of each component and the total portfolio value.Is the portfolio worth more or less than par? Why?

In [56]:
portfolio_nav = floater_value + floor_price - cap_price
print(f"cap NAV: {cap_price}")
print(f"floor NAV: {floor_price}")
print(f"portfolio NAV: {portfolio_nav}")

cap NAV: 0.9773140911242582
floor NAV: 0.24491092604572093
portfolio NAV: 99.26759683492146


The portfolio is worth less than par since we are short the more valuable option. In an par world the cap and floor should be equally priced but since we are short the cap which is priced higher the portfolio is under par.

## 2. Risk Analysis

## 2.1.
Calculate the duration of the portfolio numerically by applying a 1bp parallel shock to the spot curve.

Report the duration of each component and the portfolio.

In [103]:
SHOCK = 0.0001

# Base arrays
tenors = rate_data["tenor"].to_numpy(dtype=float)
discounts_base = rate_data["discounts"].to_numpy(dtype=float)

# --- Shock UP ---
discounts_up = discounts_base * np.exp(-SHOCK * tenors)

rate_data_up = rate_data.copy()
rate_data_up["discounts"] = discounts_up
rate_data_up["forwards"] = utils.compute_forwards(rate_data_up)

floor_strike = 0.02
cap_strike = 0.05
five_year_flat_vol = rate_data.loc[rate_data["tenor"] == 5, "flat vols"].iloc[0]

cap_price_1bp_up = utils.price_cap(rate_data_up, 5, cap_strike, implied_vol=five_year_flat_vol)
# print(f"Cap Price 1bp UP shift: {cap_price_1bp_up}")

floor_price_1bp_up = utils.price_floor(rate_data_up, 5, floor_strike, implied_vol=five_year_flat_vol)
# print(f"Floor Price 1bp UP shift: {floor_price_1bp_up}")


# --- Shock DOWN ---
discounts_dn = discounts_base * np.exp(+SHOCK * tenors)

rate_data_dn = rate_data.copy()
rate_data_dn["discounts"] = discounts_dn
rate_data_dn["forwards"] = utils.compute_forwards(rate_data_dn)

cap_price_1bp_dn = utils.price_cap(rate_data_dn, 5, cap_strike, implied_vol=five_year_flat_vol)
# print(f"Cap Price 1bp DOWN shift: {cap_price_1bp_dn}")

floor_price_1bp_dn = utils.price_floor(rate_data_dn, 5, floor_strike, implied_vol=five_year_flat_vol)
# print(f"Floor Price 1bp DOWN shift: {floor_price_1bp_dn}")

# ---- Portfolio Prices Under Shock ----
portfolio_up = floater_value + floor_price_1bp_up - cap_price_1bp_up
portfolio_dn = floater_value + floor_price_1bp_dn - cap_price_1bp_dn

# ---- Portfolio Duration (central difference) ----
portfolio_duration = (portfolio_dn - portfolio_up) / (2 * SHOCK * portfolio_nav)

In [102]:
numerical_duration_cap = -(cap_price_1bp_up - cap_price_1bp_dn) / (2 * cap_price * SHOCK)
numerical_duration_floor = -(floor_price_1bp_up - floor_price_1bp_dn) / (2 * floor_price * SHOCK)

print(f"Cap duration {numerical_duration_cap}")
print(f"Floor duration {numerical_duration_floor}")
print(f"Portfolio duration {portfolio_duration}")


Cap duration -110.63557599040213
Floor duration 104.78590491618613
Portfolio duration 1.3477602427210538


## 2.2. Calculate the OAS of the portfolio.

If the market quotes this note at par (100.00), what parallel shift to the spot curve would match that price?

Use fsolve to find the OAS.

In [ ]:
from scipy.optimize import fsolve

def price_portfolio_with_oas(oas):
    """
    Price the collared floater portfolio (+floater, +floor, -cap)
    after applying a parallel shift `oas` to the spot curve.
    
    The floater always prices at par (100) since it resets to SOFR.
    OAS shifts both discount factors and forward rates.
    """
    tenors = rate_data["tenor"].to_numpy(dtype=float)
    discounts_base = rate_data["discounts"].to_numpy(dtype=float)

    # Parallel shift: Z'(t) = Z(t) * exp(-oas * t)
    discounts_shifted = discounts_base * np.exp(-oas * tenors)

    rate_data_shifted = rate_data.copy()
    rate_data_shifted["discounts"] = discounts_shifted
    rate_data_shifted["forwards"] = utils.compute_forwards(rate_data_shifted)

    cap = utils.price_cap(rate_data_shifted, 5, cap_strike, implied_vol=five_year_flat_vol)
    floor_val = utils.price_floor(rate_data_shifted, 5, floor_strike, implied_vol=five_year_flat_vol)

    # Floater always = par; portfolio = floater + floor - cap
    return floater_value + floor_val - cap


def oas_objective(oas):
    return price_portfolio_with_oas(oas) - 100.0


oas_result = fsolve(oas_objective, x0=0.0, full_output=True)
oas = oas_result[0][0]

print(f"OAS:                     {oas*10000:.4f} bps")
print(f"Portfolio price at OAS:  {price_portfolio_with_oas(oas):.6f}")

OAS:                     -59.3472 bps
Portfolio price at OAS:  100.000000


## 2.3.
Plot the portfolio value across a range of interest rate scenarios (shock the spot curve from -300bp to +300bp).

On your plot, mark the current rate, the floor (2%), and the cap (5%).

Describe what you see in terms of convexity.